# 36120 Advanced Machine Learning - Assignment 1
## NBA Career Longevity Prediction - Experiment 1
**Student Name:** Aryan Goel  
**Goal:** Predict whether a college basketball player will achieve a sustained NBA career (>= 3 seasons in the league) using college statistics. The evaluation metric is AUPRC (Average Precision).

This notebook uses the modular preprocessing and feature engineering package `nba_prep` developed for this project.

In [ ]:
# Auto-install dependencies if they are missing in the current kernel environment
try:
    import pandas as pd
    import numpy as np
    import lightgbm as lgb
    import nba_prep
    print("All dependencies are already installed and loaded!")
except ModuleNotFoundError:
    import sys
    import subprocess
    print("Installing dependencies from requirements.txt...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "../requirements.txt"])
    print("Dependencies installed successfully! Please restart/refresh the notebook kernel.")


In [ ]:
# Imports
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score

# Import custom utility package
from nba_prep.cleaning import clean_height, clean_year, calculate_age
from nba_prep.features import (
    calculate_ast_tov_ratio,
    calculate_true_shooting_percentage,
    calculate_defensive_impact,
    calculate_scoring_efficiency,
)

### 1. Load Raw Datases

In [ ]:
train = pd.read_csv('../data/raw/train.csv', low_memory=False)
test = pd.read_csv('../data/raw/test.csv', low_memory=False)
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

### 2. Exploratory Data Analysis (EDA)
Let's look at the label distribution and top correlations.

In [ ]:
print("Target label distribution:")
print(train["label"].value_counts())
print(train["label"].value_counts(normalize=True))

In [ ]:
# Check missing values
nulls = train.isnull().mean().sort_values(ascending=False)
print("Missing value percentages (top 10 columns):")
print(nulls.head(10))

### 3. Feature Cleaning and Engineering
We apply our `nba_prep` package to clean heights (from string to inches), clean college year (to numeric 1-4 scale), calculate player age during the season, and extract complex basketball metrics.

In [ ]:
def process_dataframe(df):
    processed = df.copy()
    
    # Height & Year cleaning
    processed["ht_inches"] = processed["ht"].apply(clean_height)
    processed["yr_clean"] = processed["yr"].apply(clean_year)
    
    # Derived age
    processed["age"] = processed.apply(lambda r: calculate_age(r["year"], r["dob"]), axis=1)
    
    # Advanced metrics
    tov_est = np.where((processed["ast/tov"] > 0) & (processed["ast/tov"].notna()), processed["ast"] / processed["ast/tov"], 0.0)
    processed["ast_tov_ratio_clean"] = calculate_ast_tov_ratio(processed["ast"], tov_est)
    
    fga = processed["twoPA"] + processed["TPA"]
    processed["true_shooting_per_clean"] = calculate_true_shooting_percentage(processed["pts"], fga, processed["FTA"])
    processed["defensive_impact_score"] = calculate_defensive_impact(processed["stl"], processed["blk"], processed["dreb"])
    processed["scoring_efficiency_score"] = calculate_scoring_efficiency(processed["pts"], processed["Min_per"])
    processed["usage_ortg_interaction"] = processed["usg"] * processed["ORtg"]
    
    return processed

train_processed = process_dataframe(train)
test_processed = process_dataframe(test)
print("Finished preprocessing!")

### 4. Cross-Validation and Model Training
We use Stratified K-Fold cross-validation (5 splits) and train a LightGBM classifier. We set `class_weight='balanced'` because of the massive class imbalance.

In [ ]:
drop_cols = ["pid", "label", "ht", "yr", "dob", "type", "num"]
feature_cols = [col for col in train_processed.columns if col not in drop_cols]

X = train_processed[feature_cols]
y = train_processed["label"]
X_test = test_processed[feature_cols]

# LightGBM categorical features
categorical_cols = ["team", "conf", "role"]
for col in categorical_cols:
    if col in X.columns:
        X[col] = X[col].astype("category")
        X_test[col] = X_test[col].astype("category")

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train_processed))
test_preds = np.zeros(len(test_processed))
fold_scores = []

print("Training models...")
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.03,
        class_weight="balanced",
        random_state=42 + fold,
        verbose=-1
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(80, verbose=False)])
    
    val_preds = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_preds
    
    fold_auprc = average_precision_score(y_val, val_preds)
    fold_scores.append(fold_auprc)
    print(f"Fold {fold+1} AUPRC: {fold_auprc:.5f}")
    
    test_preds += model.predict_proba(X_test)[:, 1] / kf.n_splits

overall_auprc = average_precision_score(y, oof_preds)
print("-----------------------------------")
print(f"Mean Fold AUPRC: {np.mean(fold_scores):.5f}")
print(f"Overall OOF AUPRC: {overall_auprc:.5f}")

### 5. Final Submission Generation

In [ ]:
submission = pd.DataFrame({
    "pid": test_processed["pid"],
    "label": test_preds
})
submission.to_csv("../submission.csv", index=False)
print("Submission saved to ../submission.csv!")
print(submission.head())